In [14]:
import pandas as pd
import numpy as np

In [15]:
df = pd.read_csv("../data/raw/unique_tech_classroom_air_quality.csv")

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   timestamp                500 non-null    object 
 1   school_period            500 non-null    object 
 2   student_count_estimated  500 non-null    int64  
 3   co2_ppm                  500 non-null    float64
 4   pm25_ugm3                500 non-null    float64
 5   temperature_c            500 non-null    float64
 6   humidity_pct             500 non-null    float64
 7   robot_x_pos              500 non-null    float64
 8   robot_y_pos              500 non-null    float64
 9   ventilation_decision     500 non-null    object 
 10  air_quality_label        500 non-null    object 
dtypes: float64(6), int64(1), object(4)
memory usage: 43.1+ KB


In [17]:
df.sample(10)

,timestamp,school_period,student_count_estimated,co2_ppm,pm25_ugm3,temperature_c,humidity_pct,robot_x_pos,robot_y_pos,ventilation_decision,air_quality_label
326,2025-11-12 17:14:12,End of Day,5,518.9,5.76,20.4,41.9,0.0,2.3,IDLE,Good
258,2025-11-12 15:18:36,End of Day,7,511.7,5.54,20.5,42.1,3.8,3.7,IDLE,Good
414,2025-11-12 19:43:48,End of Day,2,461.8,5.65,20.3,41.6,0.7,3.8,IDLE,Good
85,2025-11-12 10:24:30,2nd Period,25,865.6,13.83,20.8,49.4,2.3,0.4,MONITOR,Moderate
119,2025-11-12 11:22:18,3rd Period,27,829.9,12.82,20.5,48.9,0.3,1.5,MONITOR,Moderate
228,2025-11-12 14:27:36,End of Day,6,529.2,6.83,20.5,44.6,2.8,0.8,IDLE,Good
443,2025-11-12 20:33:06,End of Day,8,490.3,7.20,20.2,41.7,1.1,0.0,IDLE,Good
114,2025-11-12 11:13:48,3rd Period,25,816.7,12.39,20.5,48.7,0.1,2.9,MONITOR,Moderate
32,2025-11-12 08:54:24,1st Period,22,770.1,10.04,20.8,45.9,0.0,0.5,MONITOR,Moderate
195,2025-11-12 13:31:30,5th Period,19,798.9,11.48,20.5,47.7,0.3,1.6,MONITOR,Moderate


## **Working with datetime**

In [18]:
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S")

df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["day"] = df["timestamp"].dt.day
df["hour"] = df["timestamp"].dt.hour
df["minute"] = df["timestamp"].dt.minute

df.drop("timestamp", axis=1, inplace=True)

In [19]:
df.sample(5)

,school_period,student_count_estimated,co2_ppm,pm25_ugm3,temperature_c,humidity_pct,robot_x_pos,robot_y_pos,ventilation_decision,air_quality_label,year,month,day,hour,minute
333,End of Day,9,521.1,5.54,20.4,41.7,1.4,0.7,IDLE,Good,2025,11,12,17,26
143,4th Period,25,866.5,12.42,20.5,49.6,3.2,1.8,MONITOR,Moderate,2025,11,12,12,3
413,End of Day,0,469.6,5.50,20.3,41.7,0.4,3.5,IDLE,Good,2025,11,12,19,42
424,End of Day,0,491.0,5.38,20.3,41.5,3.4,3.7,IDLE,Good,2025,11,12,20,0
92,Break,1,669.8,11.15,20.7,48.0,4.0,2.0,IDLE,Good,2025,11,12,10,36


In [20]:
df["air_quality_label"].value_counts()

air_quality_label
Good        335
Moderate    165
Name: count, dtype: int64

In [21]:
air_quality_mapping = {
    "Good" : 1,
    "Moderate" : 0
}

df["air_quality_label"] = df["air_quality_label"].map(air_quality_mapping)


In [22]:
df["ventilation_decision"].nunique()

2

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   school_period            500 non-null    object 
 1   student_count_estimated  500 non-null    int64  
 2   co2_ppm                  500 non-null    float64
 3   pm25_ugm3                500 non-null    float64
 4   temperature_c            500 non-null    float64
 5   humidity_pct             500 non-null    float64
 6   robot_x_pos              500 non-null    float64
 7   robot_y_pos              500 non-null    float64
 8   ventilation_decision     500 non-null    object 
 9   air_quality_label        500 non-null    int64  
 10  year                     500 non-null    int32  
 11  month                    500 non-null    int32  
 12  day                      500 non-null    int32  
 13  hour                     500 non-null    int32  
 14  minute                   5

## **Encoding**

In [24]:
from sklearn.preprocessing import LabelEncoder

def encoding(df):
    encoder = LabelEncoder()
    for col in df.columns:
        if df[col].dtype == "object":
            if df[col].nunique() <= 5:
                dummies = pd.get_dummies(df[col], prefix=col, dtype=int)
                df = pd.concat([df.drop(columns=col), dummies], axis=1)
            else:
                df[col] = encoder.fit_transform(df[col])
    return df

In [25]:
df = encoding(df)


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   school_period                 500 non-null    int32  
 1   student_count_estimated       500 non-null    int64  
 2   co2_ppm                       500 non-null    float64
 3   pm25_ugm3                     500 non-null    float64
 4   temperature_c                 500 non-null    float64
 5   humidity_pct                  500 non-null    float64
 6   robot_x_pos                   500 non-null    float64
 7   robot_y_pos                   500 non-null    float64
 8   air_quality_label             500 non-null    int64  
 9   year                          500 non-null    int32  
 10  month                         500 non-null    int32  
 11  day                           500 non-null    int32  
 12  hour                          500 non-null    int32  
 13  minut

## **Log transforms**

In [27]:
num_cols = df.select_dtypes(include=["float64", "int32", "int64"])
num_cols.skew()

school_period                  -0.946176
student_count_estimated         0.502767
co2_ppm                         0.619395
pm25_ugm3                       0.456659
temperature_c                   0.579562
humidity_pct                    0.432803
robot_x_pos                    -0.007124
robot_y_pos                     0.009703
air_quality_label              -0.725255
year                            0.000000
month                           0.000000
day                             0.000000
hour                            0.007504
minute                          0.012663
ventilation_decision_IDLE      -0.725255
ventilation_decision_MONITOR    0.725255
dtype: float64

In [28]:
import numpy as np

skewed_cols = df.select_dtypes(include="number").columns

for col in skewed_cols:
    if df[col].skew() > 0:
        df[col] = np.log1p(df[col])


In [29]:
df.sample(10)

,school_period,student_count_estimated,co2_ppm,pm25_ugm3,temperature_c,humidity_pct,robot_x_pos,robot_y_pos,air_quality_label,year,month,day,hour,minute,ventilation_decision_IDLE,ventilation_decision_MONITOR
168,7,1.945910,6.695799,2.517696,3.068053,3.914021,2.2,1.223775,0,2025,11,12,2.564949,3.828641,0,0.693147
51,0,3.178054,6.768263,2.558002,3.081910,3.899950,4.0,1.568616,0,2025,11,12,2.302585,3.295837,0,0.693147
73,1,3.258097,6.783099,2.581731,3.081910,3.910021,0.1,0.405465,0,2025,11,12,2.397895,1.609438,0,0.693147
329,6,0.000000,6.211202,2.041220,3.063391,3.756538,0.4,0.916291,1,2025,11,12,2.890372,2.995732,1,0.000000
36,0,3.295837,6.749814,2.497329,3.081910,3.864931,0.4,0.875469,0,2025,11,12,2.302585,0.693147,0,0.693147
206,4,3.178054,6.692704,2.490723,3.068053,3.895894,1.0,1.568616,0,2025,11,12,2.639057,3.931826,0,0.693147
169,7,1.386294,6.626718,2.504709,3.068053,3.906005,2.4,1.193922,0,2025,11,12,2.564949,3.871201,0,0.693147
165,3,3.218876,6.739337,2.548664,3.068053,3.921973,1.3,1.410987,0,2025,11,12,2.564949,3.713572,0,0.693147
131,2,3.135494,6.745119,2.587012,3.068053,3.919991,3.4,0.000000,0,2025,11,12,2.484907,3.761200,0,0.693147
100,2,3.178054,6.473273,2.415021,3.068053,3.867026,3.2,1.547563,1,2025,11,12,2.397895,3.931826,1,0.000000


## **Random Forest Classifier**

In [33]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, cross_val_score


x = df.drop("air_quality_label", axis=1)
y = df["air_quality_label"]

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

rfc = RandomForestClassifier()

rfc.fit(x_train, y_train)

y_pred = rfc.predict(x_test)

cr = classification_report(y_test, y_pred)

print(cr)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       1.00      1.00      1.00        67

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [ ]:
scores = cross_val_score(rfc, x,y, cv=5, scoring="accuracy")

print(scores)       
print(scores.mean())   


[0.86 1.   1.   1.   1.  ]
0.9719999999999999
0.056
